In [0]:
# Databricks notebook: Training_model

# This notebook trains a single node using BatchModelTrainer from mch.models.training.
# It is designed to be called from Main_Parallel_Runner via dbutils.notebook.run().

# ------------------------------------------------------------------------------------
# 1. Widgets: get the node name passed from Main_Parallel_Runner
# ------------------------------------------------------------------------------------
dbutils.widgets.text("only_node", "", "Node name to train")
only_node = dbutils.widgets.get("only_node")

print(f"=== Training_model started for node: {only_node} ===")

# ------------------------------------------------------------------------------------
# 2. Basic imports and path setup
# ------------------------------------------------------------------------------------
import os
import sys
import time
import json
import logging
from pathlib import Path

import mlflow
import pandas as pd

# Make sure mt-method2 repo is on sys.path
REPO_SRC = "/Workspace/9900-f18a-cake/mt-method2"
if f"{REPO_SRC}/src" not in sys.path:
    sys.path.append(f"{REPO_SRC}/src")
    print(f"[INFO] Added to sys.path: {REPO_SRC}/src")

from mch.models.training import BatchModelTrainer
from mch.config.settings import main_tree, DATA_DIR

# ------------------------------------------------------------------------------------
# 3. Environment variables for the trainer (you can tweak these if needed)
# ------------------------------------------------------------------------------------

# Only train this single node
os.environ["MCH_ONLY_NODE"] = only_node

# Default toggles / hyperparameters (use setdefault so cluster-level settings still work)
os.environ.setdefault("MCH_DISABLE_DM", "1")      # "1" = disable DM, "0" = enable DM
os.environ.setdefault("MCH_PREFILTER_TOPK", "200")
os.environ.setdefault("RF_N_ESTIMATORS", "50")
os.environ.setdefault("RF_MAX_DEPTH", "10")
os.environ.setdefault("RF_N_JOBS", "8")
os.environ.setdefault("CV_N_JOBS", "4")

print("[INFO] Environment setup for node:", only_node)
print("MCH_DISABLE_DM   =", os.environ["MCH_DISABLE_DM"])
print("MCH_PREFILTER_TOPK =", os.environ["MCH_PREFILTER_TOPK"])
print("RF_N_ESTIMATORS  =", os.environ["RF_N_ESTIMATORS"])
print("RF_MAX_DEPTH     =", os.environ["RF_MAX_DEPTH"])
print("RF_N_JOBS        =", os.environ["RF_N_JOBS"])
print("CV_N_JOBS        =", os.environ["CV_N_JOBS"])

# ------------------------------------------------------------------------------------
# 4. Optional logging configuration for this notebook
# ------------------------------------------------------------------------------------
logger = logging.getLogger("Training_model")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s")
    (DATA_DIR / "logs").mkdir(parents=True, exist_ok=True)
    fh = logging.FileHandler(str(DATA_DIR / "logs" / f"training_{only_node}.log"), encoding="utf-8")
    fh.setFormatter(fmt); fh.setLevel(logging.INFO)
    sh = logging.StreamHandler(); sh.setFormatter(fmt); sh.setLevel(logging.INFO)
    logger.addHandler(fh); logger.addHandler(sh)

logger.info("Starting BatchModelTrainer for node: %s", only_node)

# ------------------------------------------------------------------------------------
# 5. Run the training
# ------------------------------------------------------------------------------------
start_time = time.time()
status = "OK"
error_message = None
run_id = None
metrics = {}
summary_df = None

try:
    trainer = BatchModelTrainer(tree=main_tree)
    stats = trainer.train_all_models(save_dir=None, raise_on_error=True)

    node_stats = stats.get(only_node)
    if node_stats is None:
        status = "SKIPPED"
        error_message = f"No stats returned for node '{only_node}' (possibly not enough samples)."
        print(error_message)
    else:
        metrics = node_stats.get("metrics", {}) or {}
        summary_df = trainer.get_training_summary()

        print(f"=== RESULTS for node: {only_node} ===")
        print("Status      :", status)
        print("   accuracy :", metrics.get("accuracy"))
        print("   macro_f1 :", metrics.get("macro_f1"))
        print("weighted_f1 :", metrics.get("weighted_f1"))

        # --------------------------------------------------------------------------------
        # 6. Log to MLflow
        # --------------------------------------------------------------------------------
        with mlflow.start_run(run_name=f"RF_{only_node}") as run:
            run_id = run.info.run_id

            # Log metrics
            for k, v in metrics.items():
                if v is None:
                    continue
                try:
                    mlflow.log_metric(k, float(v))
                except Exception:
                    pass

            # Log some params
            mlflow.log_param("node_name", only_node)
            mlflow.log_param("n_features", node_stats.get("n_features"))
            mlflow.log_param("n_samples_total", node_stats.get("n_samples_total"))
            mlflow.log_param("MCH_DISABLE_DM", os.environ["MCH_DISABLE_DM"])
            mlflow.log_param("MCH_PREFILTER_TOPK", os.environ["MCH_PREFILTER_TOPK"])
            mlflow.log_param("RF_N_ESTIMATORS", os.environ["RF_N_ESTIMATORS"])
            mlflow.log_param("RF_MAX_DEPTH", os.environ["RF_MAX_DEPTH"])

            # Log model if estimator is available
            est = node_stats.get("estimator")
            if est is not None:
                import mlflow.sklearn
                mlflow.sklearn.log_model(est, artifact_path="model")

        print(f"MLflow run id : {run_id}")

except Exception as e:
    status = "ERROR"
    error_message = str(e)
    logger.exception("Error while training node: %s", only_node)
    print(f"[ERROR] Training failed for node {only_node}: {e}")

elapsed = time.time() - start_time
print(f"Elapsed (sec) : {elapsed:.2f}")

# ------------------------------------------------------------------------------------
# 7. Return result to Main_Parallel_Runner
# ------------------------------------------------------------------------------------
result_payload = {
    "node": only_node,
    "status": status,
    "metrics": metrics,
    "run_id": run_id,
    "elapsed_sec": elapsed,
    "error": error_message,
}

# For debugging inside this notebook
print("Result payload:", result_payload)

# Must be a string to return to dbutils.notebook.run
dbutils.notebook.exit(json.dumps(result_payload))